# LLM From DFM to Star Schema

## Import Libraries

In [ ]:
import sys, os, json, re
from datetime import datetime

sys.path.insert(0, os.path.abspath(""))

from utils.config import *
from utils.llm_client import call_llm, LLM_PROVIDER
from utils.loader import load_tables
import pandas as pd

BASE_PATH = MODELING_REPORT_DIR
LLM_PATH = MODELING_REPORT_DIR / "LLM"
os.makedirs(BASE_PATH, exist_ok=True)
LLM_PATH.mkdir(parents=True, exist_ok=True)

dfs = load_tables("cleaned", normalize_cols="lower")

schema_path = BASE_PATH / "schema.json"
assert schema_path.exists(), f"Missing: {schema_path} - run Classical Modeling first."

with open(schema_path) as f:
    classical_schema = json.load(f)


def is_llm_error(text):
    return isinstance(text, str) and text.startswith("[LLM Error]")


print("Libraries imported from utils. LLM client ready.")
print(f"LLM provider: {LLM_PROVIDER.upper()}")
print(f'  Fact:       {classical_schema["star_schema"]["fact_table"]["name"]}')
print(f'  Measures:   {classical_schema["star_schema"]["fact_table"]["measures"]}')
print(f'  Dimensions: {list(classical_schema["star_schema"]["dimension_tables"].keys())}')
print(f"  LLM output: {LLM_PATH}")

## Fact Table Suggestion

In [ ]:
def build_schema_description(dataframes: dict) -> str:
    """
    Compact textual description of the reconciled DB schema.
    Passed to the LLM as grounding context.
    """
    parts = []
    for name, df in dataframes.items():
        parts.append(f"TABLE {name}:")
        parts.append(f"  rows: {len(df):,}")
        for col in sorted(df.columns):
            dtype = str(df[col].dtype)
            n_unique = df[col].nunique()
            sample = df[col].dropna()
            sample = str(sample.iloc[0])[:30] if not sample.empty else "--"
            parts.append(f"  - {col:25s} {dtype:12s} unique={n_unique:<6d} sample={sample!r}")
        parts.append("")
    return "\n".join(parts)


schema_desc = build_schema_description(
    {
        "Flights (sample 5 rows)": dfs["Flights"].head(5),
        "Aircrafts": dfs["Aircrafts"],
        "Carriers": dfs["Carriers"],
        "Stations (sample)": dfs["Stations"].head(10),
        "Cancellation": dfs["Cancellation"],
        "Active_Weather": dfs["Active_Weather"],
        "Weather_Observations (sample 5 rows)": dfs["Weather_Observations"].head(5),
    }
)

SYSTEM_FACT = """You are a senior data warehouse modeler following the Golfarelli-Rizzi DFM methodology.

Your task is to identify the FACT TABLE in a reconciled database schema for a
US domestic flight operations Data Warehouse (year 2022).

The fact table must satisfy ALL of the following criteria:
1. Represents a clearly defined business event (not a lookup or reference table)
2. Has a fine grain - each row corresponds to one atomic occurrence of that event
3. Contains multiple numeric measures that are meaningful to aggregate
4. References multiple dimension tables via foreign keys (it is the "center" of the star)

Additional modeling hints:
- Lookup tables with few rows (< 10) and only code + description are dimension candidates, not facts
- A table with millions of rows and both FK columns and numeric columns is likely the fact
- Weather readings are context for the event, not the event itself

Respond in this exact format - no extra text:

FACT TABLE: <table_name>
GRAIN: <one precise sentence, e.g. "One row per scheduled flight departure">
REASONING:
- <specific structural reason 1>
- <specific structural reason 2>
- <specific structural reason 3>
MEASURES IDENTIFIED: <comma-separated list of numeric columns that are measures>
DIMENSIONS REFERENCED: <comma-separated list of FK columns or referenced tables>"""

prompt_fact = f"""Given this reconciled database schema for a US domestic airline project,
identify which table should be the fact table in the star schema.

Schema:
{schema_desc}

Apply the Golfarelli-Rizzi DFM criteria strictly.
Pay attention to cardinality (number of rows) and the presence of FK columns."""

response_fact = call_llm(prompt_fact, system=SYSTEM_FACT)
if is_llm_error(response_fact):
    print(f"[Fact Table Suggestion] Skipped - {response_fact}")
    response_fact = ""

print("LLM RESPONSE - Fact Suggestion:")
print()
print(response_fact)
print()
print("CLASSICAL CHOICE:")
print(f'  Fact:  {classical_schema["star_schema"]["fact_table"]["name"]}')
print(f'  Grain: {classical_schema["grain"]}')
print(f'  Match: {"Yes" if "flight" in response_fact.lower() else "REVIEW NEEDED"}')

# Save
if response_fact:
    with open(LLM_PATH / "01_fact_suggestion.txt", "w") as f:
        f.write(response_fact)
    print(f"\nSaved: {LLM_PATH / '01_fact_suggestion.txt'}")
else:
    print("Fact suggestion not saved because the LLM response failed.")

## Measure Additivity and Aggregation Rule Review

In [ ]:
measures_to_review = list(classical_schema["measures"].get("additivity", {}).keys())

SYSTEM_AGG_RULES = """You are a DW modeler specializing in dimensional modeling.

Review the additivity and meaningful aggregation rules for each measure.
Additivity is important and must be reported explicitly.

Definitions:
- additive: the measure can be summed across dimensions.
- non_additive: the measure must be recomputed or aggregated with a non-sum operator.
- semi_additive: the sum operation is meaningful across some dimensions but not others

Return ONLY valid JSON:
{
  "measures": [
    {
      "name": "...",
      "additivity": "additive|semi_additive|non_additive",
      "valid_aggs": ["SUM", "AVG", "MAX", "MIN", "RATIO"],
      "note": "one concise sentence"
    }
  ]
}"""

prompt_rules = f"""Review additivity and aggregation rules for the measures from the FLIGHTS fact table.

Fact grain: one row per scheduled flight departure (US domestic, 2022).

Classical measure metadata:
{json.dumps(classical_schema["measures"], indent=2)}

Domain context:
- Counts and total delay minutes are additive.
- Averages, maxima, minima, and rates are not additive and must be recomputed at the selected level.
- Weather measures are point-in-time readings at origin and departure hour.
- Visibility should be presented through AVG or threshold rates because isolated zero readings can dominate MIN-based visuals."""

response_rules = call_llm(prompt_rules, system=SYSTEM_AGG_RULES)
if is_llm_error(response_rules):
    print(f"[Additivity Review] Skipped - {response_rules}")
    response_rules = ""

print("LLM RESPONSE - Additivity and Aggregation Rule Review:")
print()
print(response_rules)

if response_rules:
    with open(LLM_PATH / "02_additivity_rules.json", "w") as f:
        f.write(response_rules)
    print(f"\nSaved: {LLM_PATH / '02_additivity_rules.json'}")
else:
    print("Additivity-rule response not saved because the LLM response failed.")

In [ ]:
CLASSICAL_ADDITIVITY = classical_schema["measures"].get("additivity", {})
CLASSICAL_VALID_AGGS = classical_schema["measures"].get("valid_aggs", {})

try:
    raw = re.sub(r"```json|```", "", response_rules).strip()
    llm_result = json.loads(raw)
    llm_add = {m["name"]: m.get("additivity", "") for m in llm_result.get("measures", [])}
    llm_aggs = {m["name"]: m.get("valid_aggs", []) for m in llm_result.get("measures", [])}

    print("Comparison - Classical vs LLM additivity and aggregation rules:")
    print()
    print(f'  {"Measure":<22} {"Classical Add.":<16} {"LLM Add.":<16} {"Status":<8}')
    print("  " + "-" * 72)
    for measure, additivity in CLASSICAL_ADDITIVITY.items():
        llm_type = llm_add.get(measure, "not found")
        status = "OK" if additivity == llm_type else "REVIEW"
        print(f"  {measure:<22} {additivity:<16} {llm_type:<16} {status:<8}")

    rows = []
    for measure in measures_to_review:
        rows.append(
            {
                "measure": measure,
                "classical_additivity": CLASSICAL_ADDITIVITY.get(measure, "-"),
                "llm_additivity": llm_add.get(measure, "not found"),
                "classical_valid_aggs": CLASSICAL_VALID_AGGS.get(measure, []),
                "llm_valid_aggs": llm_aggs.get(measure, []),
            }
        )
    pd.DataFrame(rows).to_csv(LLM_PATH / "02_additivity_rules_comparison.csv", index=False)
    print(f"\nSaved: {LLM_PATH / '02_additivity_rules_comparison.csv'}")

except Exception as e:
    print(f"[Parse error: {e}]")
    print("Review the raw response above manually.")

## Hierarchy Detector

In [ ]:
dim_attributes = {
    "DIM_FL_DATES": ["fl_date", "day_of_week", "is_holiday", "month", "quarter", "year"],
    # fl_date is the natural key
    # Hierarchy chain: fl_date -> month -> quarter -> year
    # Cross-dimensional attributes: day_of_week, is_holiday
    "DIM_DEP_HOURS": ["dep_hour", "time_band"],
    # dep_hour: integer 0-23; time_band groups hours
    "DIM_STATIONS": ["airport_code", "airport_name", "city", "state", "latitude", "longitude"],
    # Role-playing dimension: Origin_Station_ID and Dest_Station_ID
    # Main hierarchy: airport_code -> city -> state; coordinates are descriptive
    "DIM_CARRIERS": ["description"],
    # Role-playing dimension: MKT_Carrier_ID and OP_Carrier_ID
    "DIM_AIRCRAFTS": ["tail_num", "manufacturer", "aircraft_age"],
    # Aircraft registration dimension; manufacturer and aircraft_age are descriptive/analytical attributes
    "DIM_JUNK": ["cancellation_reason", "weather_description"],
    # Junk dimension: low-cardinality descriptors with no natural roll-up path
}

SYSTEM_HIERARCHY = """You are a DW modeler applying the Golfarelli-Rizzi DFM methodology.

For each dimension, identify the HIERARCHY:
an ordered sequence of levels from most detailed (finest grain) to most aggregated.

Rules - apply strictly:
1. Hierarchy levels must define meaningful ROLL-UP paths.
2. Descriptive attributes are properties of a dimension member, NOT hierarchy levels.
3. Boolean flags, coordinates, and derived attributes may be analytical descriptors outside the main roll-up chain.
4. If a dimension has no roll-up path, state so explicitly.
5. Note role-playing dimensions where the same physical table serves multiple roles.

For each dimension output:
HIERARCHY <DIM_NAME>:
  Natural key: <column>
  Levels: level1 -> level2 -> ... (or "single level - no roll-up path")
  Balanced: Yes / No
  Cross-dimensional attributes: <list or none>
  Descriptive attributes: <list or none>
  Missing levels suggested: <list or none>
  Role-playing note: <if applicable>"""

prompt_h = f"""Detect hierarchies in these dimension attributes for a US domestic airline DW (2022).

{json.dumps(dim_attributes, indent=2)}

Additional context:
- DIM_STATIONS is role-playing: origin and destination airport roles.
- DIM_CARRIERS is role-playing: marketing and operating carrier roles.
- DIM_AIRCRAFTS follows the approved Phase 1 design and is not part of DIM_JUNK.
- DIM_JUNK contains only cancellation and weather descriptors."""

response_h = call_llm(prompt_h, system=SYSTEM_HIERARCHY)
if is_llm_error(response_h):
    print(f"[Hierarchy Detector] Skipped - {response_h}")
    response_h = ""

print("LLM RESPONSE - Hierarchy Detector:")
print()
print(response_h)

if response_h:
    with open(LLM_PATH / "03_hierarchies.txt", "w") as f:
        f.write(response_h)
    print(f"\nSaved: {LLM_PATH / '03_hierarchies.txt'}")
else:
    print("Hierarchy response not saved because the LLM response failed.")

## Glossary Generator

In [ ]:
SYSTEM_GLOSSARY = """You are writing the official Glossary of Measures for a University
Data Warehouse exam project on US domestic airline operations.

Fact table: FLIGHTS - grain: one row per scheduled flight departure (US domestic, 2022).

For each measure, produce a compact entry and include additivity explicitly.

Output as a Markdown table with these exact columns:
| Measure | Definition | Unit | Computation | Additivity | Valid Aggs | Visualization Use |"""

glossary_measures = classical_schema.get("glossary", [])
measures_list = "\n".join(f"- {m['measure']}: {m['computation']} | additivity={m.get('additivity')} | use={m.get('visualization_use')}" for m in glossary_measures)

prompt_gloss = f"""Produce the Glossary of Measures for FLIGHTS.

Measures to document:
{measures_list}

Important constraints:
- Keep additivity explicit.
- Additive measures can be summed across dimensions.
- Non-additive measures must be recomputed or aggregated with AVG, MAX, MIN, or ratio logic.
- Visibility should avoid MIN-based visualization unless clearly justified."""

response_gloss = call_llm(prompt_gloss, system=SYSTEM_GLOSSARY)
if is_llm_error(response_gloss):
    print(f"[Glossary Generator] Skipped - {response_gloss}")
    response_gloss = ""

print("LLM GLOSSARY:")
print()
print(response_gloss)

if response_gloss:
    with open(LLM_PATH / "04_glossary.md", "w") as f:
        f.write("# Glossary of Measures - FLIGHTS\n\n")
        f.write(f"*Generated by LLM ({LLM_PROVIDER}) - review before use*\n\n")
        f.write(response_gloss)
    print(f"\nSaved: {LLM_PATH / '04_glossary.md'}")
else:
    print("Glossary not saved because the LLM response failed.")

## Schema Validator

In [ ]:
classical_summary = {
    "fact_table": classical_schema["star_schema"]["fact_table"],
    "dimension_tables": classical_schema["star_schema"]["dimension_tables"],
    "measure_rules": classical_schema["measures"],
    "grain": classical_schema["grain"],
    "special_patterns": [
        "DIM_STATIONS is a role-playing dimension: Origin_Station_ID and Dest_Station_ID both reference DIM_STATIONS(Station_ID)",
        "DIM_CARRIERS is a role-playing dimension: MKT_Carrier_ID and OP_Carrier_ID both reference DIM_CARRIERS(Carrier_ID)",
        "DIM_JUNK groups: Cancellation_Reason, Weather_Description",
        "DIM_AIRCRAFTS contains Tail_Num, Manufacturer, and Aircraft_Age as approved in the Phase 1 design",
    ],
}

SYSTEM_VALIDATE = """You are a peer reviewer of a Data Warehouse star schema built with
the Golfarelli-Rizzi DFM methodology. Your role is to provide constructive,
specific feedback - not generic advice.

Evaluate the schema on these points:
1. Fact choice and grain definition - is the grain precise and unambiguous?
2. Measure additivity and aggregation rules - are additive/non-additive classifications and valid aggregations meaningful?
3. Hierarchy completeness - are roll-up paths complete for each dimension?
4. Role-playing dimensions - are DIM_STATIONS and DIM_CARRIERS correctly modeled?
5. Junk dimension - are the grouped attributes coherent (low cardinality, no natural hierarchy)?
6. Aircraft dimension - are Tail_Num, Manufacturer, and Aircraft_Age coherently modeled in DIM_AIRCRAFTS?
7. Missing dimensions or attributes - anything important omitted?

For each finding use exactly one severity label:
- INFO: minor documentation or naming suggestion
- WARNING: design choice worth discussing with justification
- CRITICAL: clear modeling error that violates DFM principles

End with a one-line overall verdict: APPROVED / APPROVED WITH RESERVATIONS / NEEDS REVISION"""

prompt_v = f"""Peer-review this star schema for a US domestic airline flight DW (2022).

{json.dumps(classical_summary, indent=2, default=str)}

Provide specific findings with INFO / WARNING / CRITICAL labels.
End with your overall verdict."""

response_v = call_llm(prompt_v, system=SYSTEM_VALIDATE)
if is_llm_error(response_v):
    print(f"[Schema Validator] Skipped - {response_v}")
    response_v = ""

print("LLM PEER REVIEW:")
print()
print(response_v)

if response_v:
    with open(LLM_PATH / "05_peer_review.txt", "w") as f:
        f.write(response_v)
    print(f"\nSaved: {LLM_PATH / '05_peer_review.txt'}")
else:
    print("Peer review not saved because the LLM response failed.")

## Save Enriched Schema

In [ ]:
enriched = {
    "generated_at": datetime.now().isoformat(),
    "source": "8-LLM_From_DFM_to_Star_Schema.ipynb",
    "llm_provider": LLM_PROVIDER,
    "classical_schema": classical_schema,
    "llm_suggestions": {
        "provider": LLM_PROVIDER,
        "01_fact_suggestion": response_fact,
        "02_additivity_rules": response_rules,
        "03_hierarchies": response_h,
        "04_glossary": response_gloss,
        "05_peer_review": response_v,
    },
    "decision_log": [
        "Review each LLM suggestion before committing to the schema.",
        "If LLM disagrees with classical schema, document the decision and rationale.",
        "Final schema.json for the ETL phase must be the one the human modeler approves.",
        "LLM responses are advisory only - they do not replace domain validation.",
        "Additivity must remain explicit; weather measures are non-additive point-in-time readings.",
        "Role-playing dimensions (DIM_STATIONS, DIM_CARRIERS) must be confirmed correct by reviewer.",
    ],
}

enriched_path = LLM_PATH / "schema_with_llm.json"
with open(enriched_path, "w") as f:
    json.dump(enriched, f, indent=2, default=str)

print(f"schema_with_llm.json saved: {enriched_path}")
print(f"  Size: {os.path.getsize(enriched_path):,} bytes")
print()
print("LLM outputs saved in:", LLM_PATH)
print("  01_fact_suggestion.txt")
print("  02_additivity_rules.json")
print("  02_additivity_rules_comparison.csv")
print("  03_hierarchies.txt")
print("  04_glossary.md")
print("  05_peer_review.txt")
print("  schema_with_llm.json")